# Pipeline - gotowy potok przetwarzania

Potok przyjmuje dane jednego klienta i zwraca przewidywanie czy odejdzie z banku.
Używa najlepszego modelu - SVM (RBF) z zyskiem 500,326$.

In [3]:
import sys
sys.path.append("..")

import pandas as pd
import joblib
from src.preprocessing import load_data, get_preprocessed_split
model = joblib.load("../models/svm.pkl")
threshold = joblib.load("../models/svm_threshold.pkl")

df = load_data()
_, _, _, _, _, _, preprocessor = get_preprocessed_split(df)

print(f"Model wczytany. Próg decyzyjny: {threshold:.4f}")

Model wczytany. Próg decyzyjny: 0.6336


In [5]:
def predict(sample: dict) -> tuple:
    df_sample = pd.DataFrame([sample])
    X = preprocessor.transform(df_sample)
    probability = model.predict_proba(X)[0, 1]
    prediction = int(probability >= threshold)
    return prediction, probability

## Przykładowi klienci

In [6]:
klienci = [
    {
        "nazwa": "Klient A — młody, aktywny, niskie saldo",
        "dane": {
            "CreditScore": 700, "Geography": "France", "Gender": "Male",
            "Age": 28, "Tenure": 2, "Balance": 0, "NumOfProducts": 2,
            "HasCrCard": 1, "IsActiveMember": 1, "EstimatedSalary": 40000,
            "Satisfaction Score": 4, "Card Type": "SILVER", "Point Earned": 500
        }
    },
    {
        "nazwa": "Klient B — starszy, nieaktywny, wysokie saldo",
        "dane": {
            "CreditScore": 500, "Geography": "Germany", "Gender": "Female",
            "Age": 55, "Tenure": 8, "Balance": 120000, "NumOfProducts": 1,
            "HasCrCard": 0, "IsActiveMember": 0, "EstimatedSalary": 90000,
            "Satisfaction Score": 2, "Card Type": "GOLD", "Point Earned": 200
        }
    },
    {
        "nazwa": "Klient C — średni wiek, aktywny, dużo produktów",
        "dane": {
            "CreditScore": 650, "Geography": "Spain", "Gender": "Male",
            "Age": 38, "Tenure": 5, "Balance": 50000, "NumOfProducts": 3,
            "HasCrCard": 1, "IsActiveMember": 1, "EstimatedSalary": 60000,
            "Satisfaction Score": 3, "Card Type": "PLATINUM", "Point Earned": 350
        }
    }
]

for klient in klienci:
    prediction, probability = predict(klient["dane"])
    wynik = "Klient odejdzie - przyznaj zniżke" if prediction == 1 else "Klient zostaje"
    print(f"{klient['nazwa']}")
    print(f"  → {wynik} (prawdopodobieństwo: {probability:.2%})\n")

Klient A — młody, aktywny, niskie saldo
  → Klient zostaje (prawdopodobieństwo: 7.64%)

Klient B — starszy, nieaktywny, wysokie saldo
  → Klient odejdzie - przyznaj zniżke (prawdopodobieństwo: 93.39%)

Klient C — średni wiek, aktywny, dużo produktów
  → Klient odejdzie - przyznaj zniżke (prawdopodobieństwo: 95.29%)

